<a href="https://colab.research.google.com/github/shubhamkamate2005-netizen/Telcom_Churn_prediction/blob/main/Telcom_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import col, when
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [20]:
# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("ChurnPredictionPipeline") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

In [21]:
# 2. Load Data
df = spark.read.csv("churn_data.csv", header=True, inferSchema=True)

In [22]:
df.describe().show()

+-------+----------+------+------------------+-------+----------+------------------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+------------------+------------------+-----+
|summary|customerID|gender|     SeniorCitizen|Partner|Dependents|            tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|    MonthlyCharges|      TotalCharges|Churn|
+-------+----------+------+------------------+-------+----------+------------------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+------------------+------------------+-----+
|  count|      7043|  7043|              7043|   7043|      7043|     

In [23]:
df.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



In [24]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import col, when

# 3. Data Cleaning

df = df.withColumn("TotalCharges", F.trim(col("TotalCharges"))) # Trim spaces
df = df.withColumn("TotalCharges", when(col("TotalCharges") == "", "0.0").otherwise(col("TotalCharges"))) # Check for empty string after trimming
df = df.withColumn("TotalCharges", col("TotalCharges").cast(DoubleType()))

# Ensure 'Churn' column is strictly binary ('Yes' or 'No')
df = df.withColumn("Churn", when(col("Churn") == "Yes", "Yes").when(col("Churn") == "No", "No").otherwise(None))
# Drop rows where 'Churn' is now None (i.e., was not 'Yes' or 'No')
df = df.dropna(subset=["Churn"])

df = df.dropna()

In [25]:
import pyspark.sql.functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col, when

# 4. Feature Engineering

# Directly map 'Churn' to a binary numeric 'label' column (1.0 for 'Yes', 0.0 for 'No')
df = df.withColumn("label", when(col("Churn") == "Yes", 1.0).otherwise(0.0))

# Identify categorical and numerical columns
categorical_cols = [col_name for col_name, col_type in df.dtypes if col_type == 'string' and col_name != 'customerID' and col_name != 'Churn']
numerical_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

# StringIndexer for categorical columns
indexers = [
    StringIndexer(inputCol=col_name, outputCol=col_name + '_indexed', handleInvalid='keep')
    for col_name in categorical_cols
]

# VectorAssembler to combine all feature columns into a single vector
indexed_categorical_cols = [col_name + '_indexed' for col_name in categorical_cols]
# 'label' is already numeric, no need for indexing it
feature_cols = indexed_categorical_cols + numerical_cols
assembler = VectorAssembler(inputCols=feature_cols, outputCol='raw_features')

# Scale the features
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False)

# 5. Define Model
# labelCol now refers to the directly created 'label' column
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100)

# 6. Build the Pipeline
# Removed label_indexer as 'label' is now directly created
stages = indexers + [assembler, scaler, rf]
pipeline = Pipeline(stages=stages)

# 7. Train/Test Split
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# 8. Train the Model
model = pipeline.fit(train_data)

# 9. Make Predictions
predictions = model.transform(test_data)

# 10. Evaluate the Model
# Instantiate the evaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

# Calculate accuracy on the test data
accuracy = evaluator.evaluate(predictions)

# Print the result
print(f"Test Accuracy = {accuracy:.4f}")

Test Accuracy = 0.7888


In [26]:
# 10. Evaluate Model
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)
print(f"Model AUC-ROC: {auc}")

# 11. Save the Model for Production
model.write().overwrite().save("churn_prediction_pipeline_model")

# Stop spark session
spark.stop()

Model AUC-ROC: 0.8363418521480274
